# 02 — GQA + KV cache (Phase 2)

LatentServe's own execution path: our decoder layer loop, our GQA
attention, our preallocated KV cache, over the same fixed weights. HF is
demoted to correctness oracle.

Produces `results/raw/phase2_gqa.jsonl`.

**Budget:** ~60-75 minutes for all three runs. Run A alone (~20 min) is
enough to clear Gate 2 and check the predictions.

## 0. Session settings (do this in the sidebar before running anything)

* **Accelerator:** GPU T4 x2
* **Internet:** ON (needed to clone the repo and pull the checkpoint)
* **Persistence:** Files only

Everything below pins execution to **GPU 0**. Phases 0-17 are all
single-GPU experiments; the second T4 is Phase 18's subject, and letting
work drift across both would make every earlier number incomparable.

In [ ]:
import os

# Set before torch is imported anywhere, and before any `!python` child
# process inherits the environment.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Two ways in. If you attached the repo as a Kaggle Dataset, point
# UPLOAD at it; otherwise it clones. A private repo needs a PAT in
# Add-ons -> Secrets as GITHUB_TOKEN.
REPO  = "https://github.com/Jash-Vora/LatentServe.git"
DEST  = Path("/kaggle/working/LatentServe")
UPLOAD = Path("/kaggle/input/latentserve")   # optional dataset fallback

if not DEST.exists():
    if UPLOAD.exists():
        !cp -r {UPLOAD} {DEST}
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass  # public repo, or you're pulling a fork
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        subprocess.run(["git", "clone", "--depth", "1", url, str(DEST)], check=True)

os.chdir(DEST)
sys.path.insert(0, str(DEST))
os.environ["PYTHONPATH"] = f"{DEST}:" + os.environ.get("PYTHONPATH", "")
print("cwd:", Path.cwd())
!git log --oneline -1

In [ ]:
# Do NOT install torch here. Kaggle ships a build matched to its driver;
# replacing it is the fastest way to lose an afternoon.
!pip install -q pydantic pyyaml 2>&1 | tail -1

import torch, transformers
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__)

major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
if (major, minor) < (4, 40):
    print("\n[!] pyproject requires transformers>=4.40. Run:  !pip install -q -U transformers")
else:
    print("transformers OK (LatentServe's own layer loop is version-independent "
          "by design, but model/qwen.py's HF cache walk needs >=4.40)")

In [ ]:
# Pull the checkpoint once, up front, so the download (~3 GB) never lands
# inside a timed benchmark. Later sessions re-download unless you save the
# HF cache as a Kaggle Dataset — worth doing once you're iterating daily.
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

cfg = AutoConfig.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
print(f"layers={cfg.num_hidden_layers} q_heads={cfg.num_attention_heads} "
      f"kv_heads={cfg.num_key_value_heads} "
      f"group={cfg.num_attention_heads // cfg.num_key_value_heads} "
      f"head_dim={getattr(cfg, 'head_dim', None) or cfg.hidden_size // cfg.num_attention_heads}")
_ = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
_ = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype="float16")
del _
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("cached")

## 1. Gate 2 — can LatentServe execute cached decoding?

Tier 1 needs no GPU and no checkpoint (tiny random Qwen2, float32) and
runs in seconds. Tier 2 is the real thing against the Phase 1 oracle:
logits vs. `forward_teacher_forced`, identical greedy ids, cache bytes
vs. HF's measured `past_key_values`. It loads fp32 for the numerical
gates, for the same reason `test_phase1_correctness.py` does.

In [ ]:
!python -m pytest tests/test_phase2_gqa.py -v 2>&1 | tail -30

In [ ]:
!LATENTSERVE_REAL_MODEL_TESTS=1 python -m pytest tests/test_phase2_gqa.py -v -k real 2>&1 | tail -20

## 2. Run A — parity with the Phase 1 baseline (batch 1)

Same weights, same prompts, same process as the HF path
(`--include-hf-baseline`), which controls for machine state rather than
just configuration. 32K is where `--prefill-chunk-size` earns its keep:
attention cost is unchanged, peak activation memory drops from
O(context) to O(chunk).

~20 minutes.

In [ ]:
!python -m benchmarks.runners.phase2_gqa \
    --config configs/phase2_gqa.yaml \
    --context-lengths 1024 4096 8192 16384 32768 \
    --batch-sizes 1 \
    --output-tokens 128 --repeats 3 --warmup 1 \
    --prefill-chunk-size 4096 \
    --include-hf-baseline

## 3. Run B — batch scaling

Where the KV term starts to matter. At batch 1 and 4K context, KV is
~4% of the bytes a decode step reads; at batch 8 and 16K it is ~55%.
That ratio is what bounds anything MLA can do in Phase 7.

Points whose cache would exceed `--kv-budget-gb` are skipped with the
max feasible context printed, rather than OOMing mid-sweep.

~25 minutes.

In [ ]:
!python -m benchmarks.runners.phase2_gqa \
    --config configs/phase2_gqa.yaml \
    --context-lengths 4096 8192 16384 \
    --batch-sizes 1 4 8 \
    --output-tokens 128 --repeats 3 --warmup 1 \
    --prefill-chunk-size 4096 \
    --kv-budget-gb 8

## 4. Run C — the GQA-configuration experiment

`mha_sim` stores one KV head per query head instead of sharing across
the group of 6. Identical arithmetic and identical logits (there's a
test), 6x the KV memory and traffic — so model quality is held exactly
constant and any latency difference is a pure memory-system effect.

~25 minutes.

In [ ]:
!python -m benchmarks.runners.phase2_gqa \
    --config configs/phase2_gqa.yaml \
    --context-lengths 4096 8192 16384 \
    --batch-sizes 1 4 \
    --kv-heads-modes native mha_sim \
    --output-tokens 128 --repeats 3 --warmup 1 \
    --prefill-chunk-size 4096 \
    --kv-budget-gb 8

## 5. Check the predictions

`docs/phase2.md` registers five, before any of this ran. Mark each one
confirmed or falsified in writing — a falsified prediction is a result,
and usually a more interesting one.

In [ ]:
import pandas as pd

df = pd.read_json("results/raw/phase2_gqa.jsonl", lines=True)
ex = pd.json_normalize(df["extra"])
d = pd.concat([df.drop(columns=["extra"]), ex], axis=1)
ls = d[d.system == "latentserve_gqa"]

cols = ["kv_heads_mode", "batch_size", "context_length", "ttft_ms", "tpot_ms",
        "peak_vram_mb", "kv_cache_mb", "decode_kv_fraction",
        "achieved_bandwidth_gb_s", "bandwidth_utilization"]
print(ls[cols].to_string(index=False))

b1 = ls[(ls.batch_size == 1) & (ls.kv_heads_mode == "native")].sort_values("context_length")
print("\nP1 decode overhead-bound?   max achieved "
      f"{ls.achieved_bandwidth_gb_s.max():.0f} GB/s "
      f"({ls.bandwidth_utilization.max() * 100:.0f}% of peak) — falsified if >200 GB/s")
if len(b1) > 1:
    lo, hi = b1.tpot_ms.iloc[0], b1.tpot_ms.iloc[-1]
    print(f"P2 TPOT flat to long ctx?   {lo:.2f} -> {hi:.2f} ms "
          f"({(hi / lo - 1) * 100:+.0f}%) — predicted within 15%")
print(f"P5 KV share of decode bytes: {ls.decode_kv_fraction.min() * 100:.1f}% .. "
      f"{ls.decode_kv_fraction.max() * 100:.1f}% — run Phase 7 at the high end")

In [ ]:
# P3: peak VRAM vs. the Phase 1 reference, at matched context.
p1 = pd.read_json("results/raw/phase1_reference.jsonl", lines=True)
p1 = p1[p1["extra"].apply(lambda e: (e or {}).get("status", "ok")) == "ok"]
m = b1.merge(p1[["context_length", "peak_vram_mb", "tpot_ms"]],
             on="context_length", suffixes=("_ls", "_hf"))
m["vram_saved_mb"] = m.peak_vram_mb_hf - m.peak_vram_mb_ls
print(m[["context_length", "peak_vram_mb_hf", "peak_vram_mb_ls",
         "vram_saved_mb", "tpot_ms_hf", "tpot_ms_ls"]].to_string(index=False))

In [ ]:
# P4: what mha_sim actually costs, at matched batch and context.
piv = (ls.pivot_table(index=["batch_size", "context_length"],
                      columns="kv_heads_mode",
                      values=["tpot_ms", "kv_cache_mb"])
       .dropna())
if ("tpot_ms", "mha_sim") in piv.columns:
    piv[("ratio", "tpot")] = piv[("tpot_ms", "mha_sim")] / piv[("tpot_ms", "native")]
    piv[("ratio", "kv")] = piv[("kv_cache_mb", "mha_sim")] / piv[("kv_cache_mb", "native")]
    print(piv.round(2).to_string())
    print("\nKV ratio should be 6.0. A TPOT ratio far below that means decode "
          "is not KV-bandwidth-bound (P1). Far above means the KV read path is "
          "well off peak bandwidth — an Nsight Compute question for Phase 12.")

## 6. Before you close the session

* Gate 2 checklist in `docs/phase2.md` filled in.
* Each prediction marked confirmed or falsified, with the number.
* Results zipped below — `results/raw/*.jsonl` is gitignored and exists
  only as notebook output.

Phase 3 (paged KV cache) compares against exactly these contiguous-cache
numbers, so keep the file.

In [ ]:
# results/raw/*.jsonl is gitignored, so it survives only as notebook
# output. Zip it into /kaggle/working before the session ends.
!cd /kaggle/working/LatentServe && zip -qr /kaggle/working/results_phase2.zip results/raw
!ls -la /kaggle/working/*.zip